In [ ]:
import itertools
import math
import os
from collections import Counter
from itertools import combinations
from math import floor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars.selectors
import seaborn as sns
from dns.ttl import make
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from scipy.special import comb

import social_groups.polars_columns as plc
from social_groups.analysis.defs.notebooks.definitions import (
    register_materialization,
    global_notebook_registry,
)
from social_groups.analysis.polars_transformations.make_group_constellation import (
    parse_model_family,
    parse_parameters,
)
from social_groups.polars_values import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.basics import list_normalized_entropy
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)
from social_groups.reporting.plots.no_discussion_voting_by_number_of_participants import (
    no_discussion_voting_by_number_of_participants,
)

%load_ext autoreload
%autoreload 2

In [ ]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

N_SEEDS = 10

In [65]:
from social_groups.analysis.definitions import defs

no_discussion_data: pl.DataFrame = defs().load_asset_value("final_no_discussion_voting")
baseline: pl.DataFrame = defs().load_asset_value("final_baseline")

2026-06-07 14:25:11 +0200 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/final_no_discussion_voting.parquet using PolarsParquetIOManager...
2026-06-07 14:25:13 +0200 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/final_baseline.parquet using PolarsParquetIOManager...


In [67]:
no_discussion_data = (
    no_discussion_data.with_columns(
        pl.col("answers_at_beginning")
        .list.eval(parser(pl.element()))
        .alias("parsed_individual_answers"),
    )
    .with_columns(
        pl.col(plc.answer_string)
        .str.split("")
        .list.sample(12, with_replacement=True)
        .alias("answer_string_list")
    )
    .with_columns(
        compared_individual_answers=comparer(
            pl.col("parsed_individual_answers").list.explode(),
            pl.col("answer_string_list").list.explode(),
        )
        .implode()
        .over(pl.int_range(pl.len())),
    )
    .drop("answer_string_list")
)

In [ ]:
baseline_performance = (
    baseline.with_columns(
        comparer(parser(pl.col(plc.final_answer)), pl.col(plc.answer_string)).alias(
            plc.is_correct
        )
    )
    .group_by(
        [
            "use_thinking",
            "with_few_shot_prompting",
            "model_name",
            "generation_method",
        ]
    )
    .agg(
        pl.len(),
        pl.col(plc.is_correct).mean().alias(plc.accuracy),
        pl.col(plc.used_input_tokens).mean(),
        pl.col(plc.used_output_tokens).mean(),
    )
    .sort(plc.accuracy, descending=True)
    .with_columns(
        parse_model_family(pl.col("model_name")).alias(plc.model_family),
        pl.col("model_name")
        .replace(MODEL_NAME_TO_LETTER_MAPPING)
        .alias("model_letter"),
    )
    .filter(pl.col("generation_method") == "standard")
    .filter(pl.col("use_thinking") == True)
    .filter(pl.col("with_few_shot_prompting") == True)
)

assert baseline_performance["len"].unique().item() == 1000

baseline_performance = baseline_performance.drop("len")

baseline_performance

### Main Results

In [ ]:
no_discussion_data.select(
    [
        pl.all().repeat_by(12),
        pl.int_ranges(12).alias("seed"),
    ]
).explode(pl.all()).head()

In [ ]:
N = len(no_discussion_data["compared_individual_answers"].first())

pl.set_random_seed(42)


no_discussion_with_different_no_agents = (
    no_discussion_data.select(
        [
            pl.all().repeat_by(N_SEEDS),
            pl.int_ranges(N_SEEDS).alias("seed"),
        ]
    )
    .explode(polars.selectors.all())
    .with_columns(
        [
            pl.col("parsed_individual_answers")
            .list.sample(i, with_replacement=False)
            .alias(f"sampled_{i}")
            for i in range(1, N + 1)
        ],
    )
    .with_columns(
        [
            comparer(
                group_reply(pl.col(f"sampled_{i}")),
                pl.col(plc.answer_string),
            ).alias(f"accuracy_{i}")
            for i in range(1, N + 1)
        ],
    )
    .with_columns(
        [
            list_normalized_entropy(pl.col(f"sampled_{i}"), 11).alias(f"entropy_{i}")
            for i in range(1, N + 1)
        ],
    )
    .with_columns(
        [
            pl.col(plc.answers_at_beginning)
            .list.sample(i, with_replacement=True)
            .list.eval(pl.element().str.len_chars())
            .list.sum()
            .alias(f"output_chars_{i}")
            for i in range(1, N + 1)
        ]
    )
    .group_by("model_name", "temperature", "seed")
    .agg(
        pl.selectors.starts_with("accuracy_").mean(),
        pl.selectors.starts_with("output_chars_").mean(),
        pl.selectors.starts_with("entropy_").mean(),
    )
)

accuracy = no_discussion_with_different_no_agents.unpivot(
    pl.selectors.starts_with("accuracy_"),
    index=["model_name", "temperature", "seed"],
    variable_name="no_participants",
    value_name="accuracy",
).with_columns(pl.col("no_participants").str.split("_").list.last().cast(int))

output_chars = no_discussion_with_different_no_agents.unpivot(
    pl.selectors.starts_with("output_chars_"),
    index=["model_name", "temperature", "seed"],
    variable_name="no_participants",
    value_name="output_chars",
).with_columns(pl.col("no_participants").str.split("_").list.last().cast(int))

entropy = no_discussion_with_different_no_agents.unpivot(
    pl.selectors.starts_with("entropy_"),
    index=["model_name", "temperature", "seed"],
    variable_name="no_participants",
    value_name="entropy",
).with_columns(pl.col("no_participants").str.split("_").list.last().cast(int))


no_discussion_with_different_no_agents = (
    accuracy.join(
        output_chars, on=["model_name", "temperature", "no_participants", "seed"]
    )
    .join(entropy, on=["model_name", "temperature", "no_participants", "seed"])
    .with_columns(
        parse_model_family(pl.col("model_name")).alias("family"),
        parse_parameters(pl.col("model_name").alias("parameters")),
    )
)

SINGLE_MODEL_RESULTS = no_discussion_with_different_no_agents

In [ ]:
no_discussion_with_different_no_agents.group_by(
    "model_name", "temperature", "family", "parameters", "no_participants"
).agg(
    pl.col("accuracy").mean().alias("acc_mean"),
    pl.col("accuracy").min().alias("acc_min"),
    pl.col("accuracy").max().alias("acc_max"),
)

In [ ]:
SINGLE_MODEL_KNOWS = (
    no_discussion_data.with_columns(
        pl.col("compared_individual_answers")
        .map_elements(lambda x: any([a for a in x]), return_dtype=pl.Boolean)
        .alias("knows"),
    )
    .group_by("model_name", "temperature")
    .agg(
        pl.col("knows").mean().alias("knows_percentage"),
        pl.col("knows").sum().alias("knows_absolute"),
        pl.col("knows").len().alias("knows_possible_total"),
    )
)
SINGLE_MODEL_KNOWS

In [ ]:
g = no_discussion_voting_by_number_of_participants(
    no_discussion_with_different_no_agents.drop(
        "output_chars", "model_name", "entropy"
    ),
    title_base="Voting Accuracy (No Debate), 1000-subset MMLUPro",
)
register_materialization(
    "no_discussion_voting_by_number_of_participants",
    g.figure,
    "Development of performance for accuracy of no discussion voting by number of participants, 1000-subset MMLUPro, 10 seeds with min/mean/max",
)

### Entropy Relationship of the result:

In [ ]:
# assume df is a pandas DataFrame (convert from polars if needed)
# df = df.to_pandas()

df = (
    no_discussion_with_different_no_agents.group_by(
        "model_name", "temperature", "no_participants", "family", "parameters"
    )
    .agg(pl.col("accuracy").mean(), pl.col("entropy").mean())
    .with_columns(pl.col("temperature").cast(pl.Float64).round(2))
    .sort("model_name", "temperature", "no_participants", "family", "parameters")
    .rename(
        {
            "no_participants": "Participants (LLM Calls)",
            "family": "Family",
            "parameters": "Parameters",
            "entropy": "Entropy",
            "accuracy": "Accuracy",
            "temperature": "Temperature",
        }
    )
    .to_pandas()
)

sns.set_theme(
    style="whitegrid",
    context="paper",
    palette="muted",
    font_scale=1.2,
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.linestyle": "--",
        "grid.alpha": 0.3,
    },
)

g = sns.relplot(
    data=df,
    col="Temperature",
    hue="Parameters",
    kind="scatter",
    x="Entropy",
    y="Accuracy",
    style="Family",
    size="Participants (LLM Calls)",
    edgecolor="black",
    palette="viridis",
    linewidth=0.6,
    alpha=0.85,
    sizes=[i * 15 for i in range(12)],
)

g.add_legend(title="Legend", bbox_to_anchor=(1.25, 1.0), loc="upper right")


g.set_xlabels("Normalized Entropy")
g.set_ylabels("Accuracy")

register_materialization(
    "entropy_accuracy_curve_for_different_participant_numbers_general_case",
    g.figure,
    "Relation of Entropy of group answer to accuracy for different models, temperatures and participants",
)

### Diverging Characters

In [ ]:
def tokenize(text: str):
    return set(text.lower().split())


def pairwise_jaccard(answers):
    token_sets = [tokenize(a) for a in answers]
    if len(token_sets) < 2:
        return 1.0
    sims = []
    for a, b in itertools.combinations(token_sets, 2):
        if not a and not b:
            sims.append(1.0)
        else:
            sims.append(len(a & b) / max(1, len(a | b)))

    return sum(sims) / len(sims)

In [ ]:
jaccard_sim_summary = (
    no_discussion_data.group_by(["model_name", "temperature"])
    .agg(
        [
            pl.col("answers_at_beginning")
            .map_elements(pairwise_jaccard, return_dtype=pl.Float64)
            .alias("self_jaccard_similarity")
        ]
    )
    .group_by(["model_name", "temperature"])
    .agg(
        [
            pl.col("self_jaccard_similarity")
            .list.mean()
            .item()
            .alias("mean_similarity"),
            pl.col("self_jaccard_similarity").list.std().item().alias("std_similarity"),
        ]
    )
    .sort(["model_name", "temperature"])
    .with_columns(
        parse_model_family(pl.col("model_name")).alias("family"),
        parse_parameters(pl.col("model_name").alias("parameters")),
        pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING).alias("size"),
    )
    .drop("model_name")
)

jaccard_sim_summary

In [ ]:
# --- Seaborn global style (this is the real upgrade) ---
sns.set_theme(
    style="whitegrid",
    context="talk",  # bigger fonts, more “paper figure”
    palette="muted",
)

pdf = (
    jaccard_sim_summary.pivot(
        values="mean_similarity", index=["family", "size"], on="temperature"
    )
    .rename({"0.0": "sim_t0", "0.7": "sim_t07"})
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(11, 6.5))

# Muted but stable palette per family
families = pdf["family"].unique()
family_palette = sns.color_palette("muted", n_colors=len(families))
family_colors = dict(zip(families, family_palette))

# keep shape encoding but slightly refined
markers = {"L": "^", "H": "P", "M": "s"}

# --- Plot lines (slightly thicker, more “designed”) ---
for _, row in pdf.iterrows():
    ax.plot(
        [0, 1],
        [row["sim_t0"], row["sim_t07"]],
        color=family_colors[row["family"]],
        linewidth=2.2,
        alpha=0.85,
        marker=markers[row["size"]],
        markersize=8,
        markeredgewidth=0.8,
        markeredgecolor="white",
    )

# --- Axes styling ---
ax.set_xticks([0, 1])
ax.set_xticklabels(["T = 0.0", "T = 0.7"])
ax.set_ylabel("Mean similarity", fontsize=13)
ax.set_title(
    "Effect of Temperature on Similarity Across Model Families", fontsize=16, pad=12
)

# cleaner grid (research-style subtle)
ax.grid(True, which="major", axis="y", alpha=0.18)
ax.grid(False, axis="x")

# remove top/right spines for modern paper look
sns.despine(ax=ax, offset=5, trim=True)

# --- Legend (clean separation: color = family, marker = size) ---
family_handles = [
    plt.Line2D([0], [0], color=family_colors[f], lw=3, label=f) for f in families
]

size_handles = [
    plt.Line2D(
        [0], [0], marker=m, color="black", linestyle="None", markersize=8, label=s
    )
    for s, m in markers.items()
]

legend1 = ax.legend(handles=family_handles, title="Family", loc="lower left")

legend2 = ax.legend(handles=size_handles, title="Size", loc="upper right")

ax.add_artist(legend1)

plt.tight_layout()
plt.show()

## Knowledge BlackHole

Define that model "knows" that question if any of the 24 answers (temperature 0.0 and 0.7) are correct:

In [ ]:
knowledge_blackhole_df = (
    no_discussion_data.with_columns(
        pl.col("compared_individual_answers").list.any().alias("knows_this_question")
    )
    .group_by(
        [
            "question_id",
            # "original_question_id",
            # "category",
            # "question",
            # "answer_string",
            "model_name",
        ]
    )
    .agg(pl.col("knows_this_question").any())
    .with_columns(
        parse_model_family(pl.col("model_name")).alias("family"),
        parse_parameters(pl.col("model_name").alias("parameters")),
    )
    .sort("family", "parameters")
    .select(
        "knows_this_question",
        "question_id",
        (
            pl.col("family")
            + pl.lit(" (")
            + pl.col("parameters").cast(str)
            + pl.lit(" B)")
        ).alias("model_name"),
    )
)

In [ ]:
def create_coverage_heatmap(
    df: pl.DataFrame, title: str = "Model Knowledge Coverage", save_path: str = None
):
    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.size": 20,
            "axes.titlesize": 22,
            "axes.labelsize": 16,
            "figure.dpi": 300,
            "savefig.dpi": 300,
            "figure.figsize": (13, 11),
        }
    )

    colors = [
        # "#f4f434",
        # "#14f4f4",
        # "#24f4f4",
        "#f2f4f4",
        "#f4faa4",
        "#f4f4f4",
        "#061d38",
    ]

    cmap = LinearSegmentedColormap.from_list("muted_blue", colors, N=512)

    # Pivot to wide format (models × questions)
    pivot_df = df.pivot(
        values="knows_this_question",
        index="model_name",
        on="question_id",
        aggregate_function="item",
    )

    model_names = pivot_df["model_name"].to_list()
    knowledge_matrix = pivot_df.drop("model_name").to_numpy().astype(bool)

    n = len(model_names)
    coverage = np.zeros((n, n))
    sizes = knowledge_matrix.sum(axis=1)

    for i in range(n):  # A (row) - the potential superseder
        for j in range(n):  # B (column) - the reference
            if sizes[j] == 0:
                coverage[i, j] = np.nan
            else:
                intersection = np.sum(knowledge_matrix[i] & knowledge_matrix[j])
                coverage[i, j] = intersection / sizes[j]

    cov_df = pd.DataFrame(coverage, index=model_names, columns=model_names)

    # ====================== PLOTTING ======================
    fig, ax = plt.subplots(figsize=(14, 12))

    # Main heatmap
    sns.heatmap(
        cov_df,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        vmin=0,
        vmax=1,
        linewidths=0.8,
        linecolor="white",
        cbar_kws={
            "label": "Coverage Ratio  |  |A ∩ B| / |B|",
            "shrink": 0.8,
            "format": "%.2f",
        },
        ax=ax,
    )

    # Beautification
    ax.set_title(
        "Knowledge Overlap between different Models",
        fontsize=18,
        pad=25,
        fontweight="bold",
    )
    ax.set_xlabel("Model B (Knowledge being covered)", fontsize=13, labelpad=15)
    ax.set_ylabel("Model A (Potential superseder)", fontsize=13, labelpad=15)

    # Rotate labels for readability
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    sns.despine(left=False, bottom=False)

    return fig

In [ ]:
register_materialization(
    "knowledge_blackhole_heatmap",
    create_coverage_heatmap(knowledge_blackhole_df),
    "Overlap of questions that can be answered correctly by each model. A model is considered 'knowing' the answer if it was able to give the correct answer at least once in 12 calls of 0.0 temperature and 12 calls of 0.7 temperature. Advanced Parsing, used on 1000-subset MMLUPro-Dataset.",
    force=True,
)


print("")

### Voting Accuracy for mixing model sizes:

In [61]:
repeated_cols = [
    "question_id",
    "answer_string",
    "temperature",
    "family",
    "parameters",
    "id",
    "parsed_individual_answers",
    "compared_individual_answers",
    "model_name",
    "used_input_tokens",
    "used_output_tokens",
]

per_question_per_model = (
    no_discussion_data.with_columns(
        parse_model_family(pl.col("model_name")).alias("family"),
        parse_parameters(pl.col("model_name").alias("parameters")),
    )
    .select(repeated_cols)
    .with_columns(
        pl.col("parameters")
        .rank(method="dense")  # or use quantiles if you prefer
        .over("family")
        .map_elements(lambda x: {1: "L", 2: "M", 3: "H"}[x], return_dtype=pl.String)
        .alias("size")
    )
)
per_question_per_model.head()

question_id,answer_string,temperature,family,parameters,id,parsed_individual_answers,compared_individual_answers,model_name,used_input_tokens,used_output_tokens,size
i64,str,f32,str,f64,i64,list[str],list[bool],str,i64,i64,str
28,"""I""",0.7,"""Qwen3.5""",9.0,109510,"[""___not_parsable___"", ""I"", … ""___not_parsable___""]","[false, true, … false]","""Qwen/Qwen3.5-9B""",17052,49152,"""H"""
23,"""C""",0.7,"""Qwen3.5""",9.0,109511,"[""C"", ""C"", … ""C""]","[true, true, … true]","""Qwen/Qwen3.5-9B""",13752,47101,"""H"""
74,"""A""",0.7,"""Qwen3.5""",9.0,109512,"[""A"", ""A"", … ""A""]","[true, true, … true]","""Qwen/Qwen3.5-9B""",21540,40772,"""H"""
81,"""C""",0.7,"""Qwen3.5""",9.0,109513,"[""C"", ""C"", … ""C""]","[true, true, … true]","""Qwen/Qwen3.5-9B""",15708,23030,"""H"""
26,"""J""",0.7,"""Qwen3.5""",9.0,109514,"[""J"", ""J"", … ""J""]","[true, true, … true]","""Qwen/Qwen3.5-9B""",18516,49047,"""H"""


In [ ]:
# Get unique (family, size, parameters)
model_info = (
    per_question_per_model.select("family", "size", "parameters")
    .unique()
    .sort(["family", "parameters"])
)

# Generate combinations


combos = []
for family in model_info["family"].unique():
    fam_df = model_info.filter(pl.col("family") == family)
    sizes = fam_df["size"].to_list()
    params = fam_df["parameters"].to_list()

    # 2-member combos
    for comb in combinations(range(len(sizes)), 2):
        idx1, idx2 = comb
        if sizes[idx1] != sizes[idx2]:  # no duplicate sizes
            group_name = "".join(sorted([sizes[idx1], sizes[idx2]]))
            combos.append(
                {
                    "family": family,
                    "group": group_name,
                    "size1": sizes[idx1],
                    "size2": sizes[idx2],
                    "param1": params[idx1],
                    "param2": params[idx2],
                }
            )

    # 3-member combo (only one possible: LMH)
    if len(sizes) >= 3:
        group_name = "LMH"
        combos.append(
            {
                "family": family,
                "group": group_name,
                "size1": "L",
                "size2": "M",
                "size3": "H",
                "param1": params[0],
                "param2": params[1],
                "param3": params[2],
            }
        )

combo_df = pl.DataFrame(combos)

In [ ]:
N = len(no_discussion_data["compared_individual_answers"].first())

mixed_sizes_expanded = (
    combo_df.join(per_question_per_model.drop("model_name"), on=["family"], how="inner")
    .filter(
        pl.when(pl.col("group").str.contains(x).not_())
        .then(pl.col("size") == x)
        .otherwise(pl.lit(False))
        .not_()
        for x in ("L", "M", "H")
    )
    .group_by(["family", "group", "question_id", "answer_string", "temperature"])
    .agg(
        # Matrix for parsed_individual_answers: list of lists → will be (n_participants x N)
        pl.col("parsed_individual_answers").implode().alias("parsed_matrix"),
        # Same for compared
        pl.col("compared_individual_answers").implode().alias("compared_matrix"),
        # Metadata
        pl.col("parameters").implode().alias("parameters_list"),
        pl.col("size").implode().alias("sizes"),
    )
)

mixed_sizes_result = (
    mixed_sizes_expanded.with_columns(seed=pl.lit(1))
    .with_row_index()  # keep track of original rows
    .with_columns(pl.lit(list(range(N_SEEDS))).alias("seed"))
    .explode("seed")
    .explode("parsed_matrix")  # now one row per participant
    .with_columns(
        pl.col("parsed_matrix")
        .list.sample(n=i, with_replacement=False)
        .alias(f"sampled_{i}")
        for i in range(1, N + 1)
    )
    .group_by("index", "seed")  # group back per original row
    .agg(
        [
            pl.col(col).unique().item()
            for col in (
                "family",
                "group",
                "question_id",
                "answer_string",
                "parameters_list",
                "temperature",
                "sizes",
            )
        ]
        + [
            pl.col(f"sampled_{i}").list.explode().alias(f"sampled_answers_{i}")
            for i in range(1, N + 1)
        ],
    )
    .drop("index")
    .with_columns(
        [
            comparer(
                group_reply(pl.col(f"sampled_answers_{i}")),
                pl.col(plc.answer_string),
            ).alias(f"accuracy_{i}")
            for i in range(1, N + 1)
        ],
    )
    .group_by("family", "group", "parameters_list", "temperature", "sizes", "seed")
    .agg(
        pl.selectors.starts_with("accuracy_").mean(),
    )
    .unpivot(
        pl.selectors.starts_with("accuracy_"),
        index=["family", "group", "parameters_list", "temperature", "sizes", "seed"],
        variable_name="no_participants",
        value_name="accuracy",
    )
    .with_columns(
        pl.col("no_participants").str.split("_").list.last().cast(int)
        * pl.col("group").str.len_chars()
    )
)

MIXED_SIZES_RESULTS = mixed_sizes_result
mixed_sizes_result.sort("accuracy", descending=True).head()

In [ ]:
MIXED_SIZES_KNOWS = (
    mixed_sizes_expanded.with_columns(
        pl.col("compared_matrix")
        .map_elements(lambda x: any([a for e in x for a in e]), return_dtype=pl.Boolean)
        .alias("knows")
    )
    .group_by("family", "group", "temperature", "sizes")
    .agg(
        pl.col("knows").mean().alias("knows_percentage"),
        pl.col("knows").sum().alias("knows_absolute"),
        pl.col("knows").len().alias("knows_possible_total"),
    )
)
MIXED_SIZES_KNOWS

### Voting Accuracy for mixing model families:

In [ ]:
# Unique models per size
model_info = (
    per_question_per_model.select(["family", "size", "parameters"])
    .unique()
    .sort(["size", "family", "parameters"])
)

# We'll store combinations
combos = []

for size in ["L", "M", "H"]:
    # All families that have this size
    models = model_info.filter(pl.col("size") == size)
    families = models["family"].to_list()
    params = models["parameters"].to_list()
    fam_to_param = dict(zip(families, params))

    if len(families) < 2:
        continue  # need at least 2 families for a group

    # === 2-member groups ===
    for comb in combinations(range(len(families)), 2):
        f1, f2 = families[comb[0]], families[comb[1]]
        group_name = size * 2  # "LL", "MM", or "HH"

        combos.append(
            {
                "group": f"{group_name}_{f1}{f2}",
                "size": size,
                "num_participants": 2,
                "family1": f1,
                "family2": f2,
                "param1": fam_to_param[f1],
                "param2": fam_to_param[f2],
            }
        )

    # === 3-member groups ===
    if len(families) >= 3:
        for comb in combinations(range(len(families)), 3):
            f1, f2, f3 = [families[i] for i in comb]
            group_name = size * 3  # "LLL", "MMM", "HHH"

            combos.append(
                {
                    "group": f"{group_name}_{f1}{f2}{f3}",
                    "size": size,
                    "num_participants": 3,
                    "family1": f1,
                    "family2": f2,
                    "family3": f3,
                    "param1": fam_to_param[f1],
                    "param2": fam_to_param[f2],
                    "param3": fam_to_param[f3],
                }
            )

different_families_combos = pl.DataFrame(combos)
different_families_combos

In [ ]:
(
    different_families_combos.unpivot(
        index=["group", "size", "num_participants"],
        on=[f"family{i}" for i in range(1, 4)],
        variable_name="slot",
        value_name="family",
    )
    .drop_nulls()
    .join(per_question_per_model, on=["family"], how="inner", validate="m:m")
    .filter(pl.col("size") == pl.col("size_right"))
    .drop("size_right")
    .group_by(["group", "question_id", "answer_string", "temperature"])
    .agg(
        pl.col("parsed_individual_answers").implode().alias("parsed_matrix"),
        pl.col("family").implode().alias("families"),
        pl.col("compared_individual_answers").implode().alias("compared_matrix"),
        pl.col("parameters").implode().alias("parameters"),
        pl.col("size").implode().alias("sizes"),
    )
).head()

In [ ]:
N = len(no_discussion_data["compared_individual_answers"].first())

mixed_families_expanded = (
    different_families_combos.unpivot(
        index=["group", "size", "num_participants"],
        on=[f"family{i}" for i in range(1, 4)],
        variable_name="slot",
        value_name="family",
    )
    .drop_nulls()
    .join(per_question_per_model, on=["family"], how="inner", validate="m:m")
    .filter(pl.col("size") == pl.col("size_right"))
    .drop("size_right")
    .group_by(["group", "question_id", "answer_string", "temperature"])
    .agg(
        pl.col("parsed_individual_answers").implode().alias("parsed_matrix"),
        pl.col("family").implode().alias("families"),
        pl.col("compared_individual_answers").implode().alias("compared_matrix"),
        pl.col("parameters").implode().alias("parameters"),
        pl.col("size").implode().alias("sizes"),
    )
)
mixed_families_result = (
    mixed_families_expanded.with_columns(seed=pl.lit(1))
    .with_row_index()  # keep track of original rows
    .with_columns(pl.lit(list(range(N_SEEDS))).alias("seed"))
    .explode("seed")
    .explode("parsed_matrix")  # now one row per participant
    .with_columns(
        pl.col("parsed_matrix")
        .list.sample(n=i, with_replacement=False)
        .alias(f"sampled_{i}")
        for i in range(1, N + 1)
    )
    .group_by("index", "seed")  # group back per original row
    .agg(
        [
            pl.col(col).unique().item()
            for col in (
                "families",
                "group",
                "question_id",
                "answer_string",
                "parameters",
                "temperature",
                "sizes",
            )
        ]
        + [
            pl.col(f"sampled_{i}").list.explode().alias(f"sampled_answers_{i}")
            for i in range(1, N + 1)
        ],
    )
    .drop("index")
    .with_columns(
        [
            comparer(
                group_reply(pl.col(f"sampled_answers_{i}")),
                pl.col(plc.answer_string),
            ).alias(f"accuracy_{i}")
            for i in range(1, N + 1)
        ],
    )
    .group_by(
        "families",
        "group",
        "parameters",
        "temperature",
        "sizes",
        "seed",
    )
    .agg(
        pl.selectors.starts_with("accuracy_").mean(),
    )
    .unpivot(
        pl.selectors.starts_with("accuracy_"),
        index=[
            "families",
            "group",
            "parameters",
            "temperature",
            "sizes",
            "seed",
        ],
        variable_name="no_participants",
        value_name="accuracy",
    )
    .with_columns(
        pl.col("no_participants").str.split("_").list.last().cast(int)
        * pl.col("families").list.len()
    )
)
MIXED_FAMILIES_RESULTS = mixed_families_result

mixed_families_result.head()

In [ ]:
mixed_families_result.with_columns(parameters=pl.col("group")).with_columns(
    parameters=pl.col("families")
    .list.eval(pl.element().str.slice(0, 1) + pl.element().str.slice(-1))
    .list.join("-"),
    family=pl.col("group").str.split("_").list.first(),
).drop("sizes", "group", "families").head()

In [ ]:
g = no_discussion_voting_by_number_of_participants(
    mixed_families_result.with_columns(parameters=pl.col("group"))
    .with_columns(
        parameters=pl.col("families")
        .list.eval(pl.element().str.slice(0, 1) + pl.element().str.slice(-1))
        .list.join("-"),
        family=pl.col("group").str.split("_").list.first(),
    )
    .drop("sizes", "group", "families"),
    title_base="Voting Accuracy (No Debate, Mixed Families)",
    figsize=(12, 7),
)

register_materialization(
    "no_discussion_voting_mixed_model_families",
    g.figure,
    "Development of performance for accuracy of no discussion voting by number of participants for mixing of different families, 1000-subset MMLUPro, 10 seeds with min/mean/max",
)
print("")

In [ ]:
MIXED_FAMILIES_KNOWS = (
    mixed_families_expanded.with_columns(
        pl.col("compared_matrix")
        .map_elements(lambda x: any([a for e in x for a in e]), return_dtype=pl.Boolean)
        .alias("knows")
    )
    .group_by("families", "group", "temperature")
    .agg(
        pl.col("knows").mean().alias("knows_percentage"),
        pl.col("knows").sum().alias("knows_absolute"),
        pl.col("knows").len().alias("knows_possible_total"),
    )
)
MIXED_FAMILIES_KNOWS

## COMPARISON for 12 Participants

In [ ]:
optimal_knows_per_method = (
    pl.concat(
        [
            MIXED_FAMILIES_KNOWS.with_columns(
                method=pl.lit("mixed_families"),
                sizes=pl.col("group").str.split("_").list.first().str.split(""),
            ).drop("group"),
            MIXED_SIZES_KNOWS.with_columns(
                method=pl.lit("mixed_sizes"),
                families=pl.col("family")
                .cast(pl.List(pl.String))
                .list.sample(pl.col("sizes").list.len(), with_replacement=True),
            ).drop("group", "family"),
            SINGLE_MODEL_KNOWS.with_columns(
                families=parse_model_family(pl.col("model_name")).cast(
                    pl.List(pl.String)
                ),
                method=pl.lit("single_model"),
                sizes=pl.col("model_name")
                .replace(MODEL_NAME_TO_LETTER_MAPPING)
                .cast(pl.List(pl.String)),
            ).drop("model_name"),
        ],
        how="diagonal_relaxed",
    )
    .with_columns(empirically_knows=pl.col("knows_percentage"))
    .drop("knows_absolute", "knows_possible_total", "knows_percentage")
)

method_comparison = (
    pl.concat(
        [
            MIXED_FAMILIES_RESULTS.with_columns(method=pl.lit("mixed_families")).drop(
                "group"
            ),
            MIXED_SIZES_RESULTS.with_columns(
                method=pl.lit("mixed_sizes"),
                families=pl.col("family")
                .cast(pl.List(pl.String))
                .list.sample(pl.col("sizes").list.len(), with_replacement=True),
                parameters=pl.col("parameters_list"),
            ).drop("parameters_list", "group", "family"),
            SINGLE_MODEL_RESULTS.with_columns(
                method=pl.lit("single_model"),
                sizes=pl.col("model_name")
                .replace(MODEL_NAME_TO_LETTER_MAPPING)
                .cast(pl.List(pl.String)),
                families=pl.col("family").cast(pl.List(pl.String)),
                parameters=pl.col("parameters").cast(pl.List(pl.Float64)),
            ).drop("output_chars", "model_name", "family"),
            baseline_performance.with_columns(
                method=pl.lit("baseline"),
                sizes=pl.col("model_letter").cast(pl.List(pl.String)),
                families=pl.col("model_family").cast(pl.List(pl.String)),
                no_participants=pl.lit(1),
                temperature=pl.lit(0.0),
                parameters=parse_parameters(pl.col("model_name")).cast(
                    pl.List(pl.Float64)
                ),
            ).drop(
                "used_input_tokens",
                "used_output_tokens",
                "model_family",
                "model_letter",
                "generation_method",
                "model_name",
                "with_few_shot_prompting",
                "use_thinking",
            ),
        ],
        how="diagonal_relaxed",
    )
    .group_by(
        "method", "families", "sizes", "temperature", "no_participants", "parameters"
    )
    .agg(
        pl.col("accuracy").mean().alias("accuracy_mean"),
        pl.col("accuracy").std().alias("accuracy_std"),
    )
    .join(
        optimal_knows_per_method,
        on=["method", "families", "sizes", "temperature"],
        how="left",
        nulls_equal=True,
    )
)
# .filter(pl.col("temperature") == 0.7).drop("temperature")
method_comparison.filter(pl.col("method") != "mixed_families")

In [ ]:
# Example:
df = method_comparison

single_lookup = (
    df.filter(pl.col("method") == "single_model")
    .with_columns(
        family=pl.col("families").list.item(),
        size=pl.col("sizes").list.item(),
    )
    .select(
        "family",
        "size",
        "temperature",
        "no_participants",
        pl.col("accuracy_mean").alias("single_accuracy"),
    )
)

baseline_lookup = (
    df.filter(pl.col("method") == "single_model")
    .filter(pl.col("no_participants") == 1)
    .with_columns(
        family=pl.col("families").list.item(),
        size=pl.col("sizes").list.item(),
    )
    .select(
        "family",
        "size",
        "temperature",
        pl.col("accuracy_mean").alias("baseline_accuracy"),
    )
)

mixed = (
    df.filter(pl.col("method").is_in(["mixed_families", "mixed_sizes", "single_model"]))
    .with_row_index("row_id")
    .explode(["families", "sizes"])
    .rename(
        {
            "families": "family",
            "sizes": "size",
        }
    )
)


best_values = (
    mixed.join(
        single_lookup,
        on=[
            "family",
            "size",
            "temperature",
            "no_participants",
        ],
        how="left",
    )
    .join(
        baseline_lookup,
        on=[
            "family",
            "size",
            "temperature",
        ],
        how="left",
    )
    .group_by("row_id")
    .agg(
        [
            pl.max("single_accuracy").alias("best_single"),
            pl.max("baseline_accuracy").alias("best_baseline"),
        ]
    )
)
result = (
    df.filter(pl.col("method").is_in(["mixed_families", "mixed_sizes", "single_model"]))
    .with_row_index("row_id")
    .join(best_values, on="row_id", how="left")
    .drop("row_id")
)

### --------------- This fills the singl_baselines that are null with the one fro 12 participants
# df = result.with_row_index("row_id")
#
# match_cols = [
#     c
#     for c in df.columns
#     if c
#     not in [
#         "best_single",
#         "no_participants",
#         "row_id",
#         "accuracy_mean",
#         "accuracy_std",
#         "empirically_knows",
#     ]
# ]
#
# # Create lookup table from rows where no_participants == 12
#
# lookup = df.filter(pl.col("no_participants") == 12).select(
#     match_cols + [pl.col("best_single").alias("fill_best_single")]
# )
#
# result = (
#     df.join(lookup, on=match_cols, how="left")
#     .with_columns(
#         pl.when(pl.col("best_single").is_null())
#         .then(pl.col("fill_best_single"))
#         .otherwise(pl.col("best_single"))
#         .alias("best_single")
#     )
#     .drop("fill_best_single")
#     .sort("row_id")
#     .drop("row_id")
# )

result

In [ ]:
from social_groups.reporting.plots.no_discussion_voting_baseline_comparison_plot import (
    no_discussion_voting_baseline_comparison_plot,
)


register_materialization(
    "mixed_methods_vs_best_constituent_models_temp00",
    no_discussion_voting_baseline_comparison_plot(
        result, 0.0, {4, 6, 12}, (-0.2, 0.05), (-0.2, 0.15)
    ).figure,
    "mixed methods compared by improvement to baseline and single-model best performance at temperature 0.0",
    force=True,
)


register_materialization(
    "mixed_methods_vs_best_constituent_models_temp07",
    no_discussion_voting_baseline_comparison_plot(
        result, 0.7, {4, 6, 12}, (-0.2, 0.05), (-0.2, 0.15)
    ).figure,
    "mixed methods compared by improvement to baseline and single-model best performance at temperature 0.7",
    force=True,
)

In [ ]:
result_with_cost = (
    result.with_columns(different_models=pl.col("parameters").list.len())
    .with_row_index("row_id")
    .explode("parameters")
    .with_columns(
        (
            pl.col("parameters")
            * 2
            * pl.col("no_participants")
            / pl.col("different_models")
        ).alias("call_cost")
    )
    .group_by("row_id")
    .agg(
        pl.exclude("row_id", "parameters", "call_cost").unique().item(),
        pl.col("parameters").implode(),
        pl.col("call_cost").sum(),
    )
    .drop("row_id")
)

In [ ]:
mixed = (
    result_with_cost.filter(
        pl.col("method").is_in(["mixed_families", "mixed_sizes", "single_model"])
    )
    .with_row_index("row_id")
    .explode(["families", "sizes"])
    .rename(
        {
            "families": "family",
            "sizes": "size",
        }
    )
)

single_lookup = (
    mixed.filter(pl.col("method") == "single_model")
    .select(
        [
            "family",
            "size",
            "temperature",
            "call_cost",
            pl.col("accuracy_mean").alias("best_cost_baseline"),
        ]
    )
    .sort("call_cost")
)

result_with_cost_baseline = (
    mixed.sort(["family", "size", "temperature", "call_cost"])
    .join_asof(
        single_lookup.sort(["family", "size", "temperature", "call_cost"]),
        on="call_cost",
        by=["family", "size", "temperature"],
        strategy="backward",
    )
    .group_by("row_id")
    .agg(
        [
            pl.max("best_cost_baseline").alias("best_cost_baseline"),
        ]
    )
)

result_with_cost_baseline = (
    result_with_cost.filter(
        pl.col("method").is_in(["mixed_families", "mixed_sizes", "single_model"])
    )
    .with_row_index("row_id")
    .join(result_with_cost_baseline, on="row_id", how="left")
    .drop("row_id")
)

register_materialization(
    "no_discussion_voting_main_result_table",
    result_with_cost_baseline,
    "Main result for no discussion voting",
    force=True,
)

In [ ]:
register_materialization(
    "mixed_methods_vs_best_constituent_models_equal_or_less_cost_temp00",
    no_discussion_voting_baseline_comparison_plot(
        result_with_cost_baseline.with_columns(
            best_single=pl.col("best_cost_baseline")
        ),
        0.0,
        {4, 6, 12},
        (-0.2, 0.05),
        (-0.2, 0.15),
        xlabel="Improvement over best single-model (same or lower cost ('2*params.' per call))",
    ).figure,
    "mixed methods compared by improvement to baseline and single-model best performance at equal or less cost at temperature 0.0",
    force=True,
)


register_materialization(
    "mixed_methods_vs_best_constituent_models_equal_or_less_cost_temp07",
    no_discussion_voting_baseline_comparison_plot(
        result_with_cost_baseline.with_columns(
            best_single=pl.col("best_cost_baseline")
        ),
        0.7,
        {4, 6, 12},
        (-0.2, 0.05),
        (-0.2, 0.15),
        xlabel="Improvement over best single-model (same or lower cost ('2*params.' per call) )",
    ).figure,
    "mixed methods compared by improvement to baseline and single-model best performance  at equal or less cost  at temperature 0.7",
    force=True,
)
print("")

## All majority vote combinations until MAD4 for mixed sizes

In [ ]:
g = no_discussion_voting_by_number_of_participants(
    mixed_sizes_result.with_columns(parameters=pl.col("group")).drop(
        "sizes", "parameters_list", "group"
    ),
    title_base="Voting Accuracy (No Debate, Mixed Model Sizes)",
)

register_materialization(
    "no_discussion_voting_mixed_model_sizes",
    g.figure,
    "Development of performance for accuracy of no discussion voting by number of participants for mixing of different model_sizes, 1000-subset MMLUPro, 10 seeds with min/mean/max",
)

In [49]:
# Get unique (family, size, parameters)
model_info = (
    per_question_per_model.select("family", "size", "parameters", "model_name")
    .unique()
    .sort(["family", "parameters"])
)

In [51]:
from itertools import combinations_with_replacement

combos = []
for family in model_info["family"].unique():
    fam_df = model_info.filter(pl.col("family") == family)
    sizes = fam_df["size"].to_list()
    params = fam_df["parameters"].to_list()
    names = fam_df["model_name"].to_list()
    size_to_param = dict(zip(sizes, params))
    for r in range(2, 5):
        for comb in combinations_with_replacement(zip(names, sizes), r):
            size_comb = [x[1] for x in comb]
            name_comb = [x[0] for x in comb]
            row = {
                "family": family,
                "group": "".join(sorted(size_comb)),
                "model_names": name_comb,
            }
            for i, size in enumerate(size_comb, start=1):
                row[f"size{i}"] = size
                row[f"param{i}"] = size_to_param[size]
            combos.append(row)
combo_df = pl.DataFrame(combos)

In [70]:
import random

simulated_rows = []

df = per_question_per_model.filter(temperature=0).drop(
    "compared_individual_answers", "temperature", "id"
)

for combo in combo_df.iter_rows(named=True):
    family = combo["family"]
    group = combo["group"]
    model_names = combo["model_names"]
    combo_sizes = [
        combo[col]
        for col in ["size1", "size2", "size3", "size4"]
        if col in combo and combo[col] is not None
    ]
    size_counts = Counter(combo_sizes)
    n_members = len(combo_sizes)
    participant_levels = list(range(n_members, 13, n_members))
    fam_df = df.filter(pl.col("family") == family)

    for qid in fam_df["question_id"].unique():
        q_df = fam_df.filter(pl.col("question_id") == qid)
        answer_pools = {}
        for size in size_counts:
            answer_pools[size] = {
                "answers": (
                    q_df.filter(pl.col("size") == size)["parsed_individual_answers"]
                    .item()
                    .to_list()
                ),
                "total_input_tokens": (
                    q_df.filter(pl.col("size") == size)["used_input_tokens"].item()
                ),
                "total_output_tokens": (
                    q_df.filter(pl.col("size") == size)["used_output_tokens"].item()
                ),
            }

        answer_string = q_df["answer_string"].unique().item()

        for seed in range(N_SEEDS):
            rng = random.Random(seed)

            for no_participants in participant_levels:
                draws_per_member = no_participants // n_members
                responses = []
                input_tokens_sum = 0
                output_tokens_sum = 0

                for size, n_members_of_size in size_counts.items():
                    n_draws = draws_per_member * n_members_of_size
                    pool = answer_pools[size]
                    sampled = rng.sample(pool["answers"], k=n_draws)
                    responses.extend(sampled)
                    input_tokens_sum += (
                        n_draws * pool["total_input_tokens"] / len(pool["answers"])
                    )
                    output_tokens_sum += (
                        n_draws * pool["total_output_tokens"] / len(pool["answers"])
                    )

                simulated_rows.append(
                    {
                        "question_id": qid,
                        "family": family,
                        "group": group,
                        "seed": seed,
                        "no_participants": no_participants,
                        "responses": responses,
                        "answer_string": answer_string,
                        "model_names": model_names,
                        "used_input_tokens": input_tokens_sum,
                        "used_output_tokens": output_tokens_sum,
                    }
                )
sim_df = (
    pl.from_dicts(simulated_rows)
    .with_columns(pl.col("group").str.split("").alias("sizes"))
    .with_columns(
        comparer(
            group_reply(pl.col("responses")),
            pl.col(plc.answer_string),
        ).alias("accuracy")
    )
)
del simulated_rows

In [71]:
sim_df.head(20)

question_id,family,group,seed,no_participants,responses,answer_string,model_names,used_input_tokens,used_output_tokens,sizes,accuracy
i64,str,str,i64,i64,list[str],str,list[str],f64,f64,list[str],bool
0,"""Ministral""","""LL""",0,2,"[""H"", ""H""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",3140.0,762.333333,"[""L"", ""L""]",true
0,"""Ministral""","""LL""",0,4,"[""H"", ""___not_parsable___"", … ""E""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",6280.0,1524.666667,"[""L"", ""L""]",true
0,"""Ministral""","""LL""",0,6,"[""H"", ""___not_parsable___"", … ""H""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",9420.0,2287.0,"[""L"", ""L""]",true
0,"""Ministral""","""LL""",0,8,"[""H"", ""___not_parsable___"", … ""H""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",12560.0,3049.333333,"[""L"", ""L""]",true
0,"""Ministral""","""LL""",0,10,"[""H"", ""___not_parsable___"", … ""H""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",15700.0,3811.666667,"[""L"", ""L""]",true
…,…,…,…,…,…,…,…,…,…,…,…
0,"""Ministral""","""LL""",2,8,"[""H"", ""H"", … ""H""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",12560.0,3049.333333,"[""L"", ""L""]",true
0,"""Ministral""","""LL""",2,10,"[""H"", ""H"", … ""H""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",15700.0,3811.666667,"[""L"", ""L""]",true
0,"""Ministral""","""LL""",2,12,"[""H"", ""H"", … ""H""]","""H""","[""mistralai/Ministral-3-3B-Reasoning-2512"", ""mistralai/Ministral-3-3B-Reasoning-2512""]",18840.0,4574.0,"[""L"", ""L""]",true


In [74]:
all_group_combinations_majority_voting = (
    sim_df.group_by(
        "family", "group", "no_participants", "sizes", "model_names", "seed"
    )
    .agg(
        pl.col("accuracy").mean(),
        pl.col("used_input_tokens").mean(),
        pl.col("used_output_tokens").mean(),
    )
    .group_by("family", "group", "no_participants", "sizes", "model_names")
    .agg(
        pl.col("accuracy").mean().alias("accuracy_mean"),
        pl.col("accuracy").std().alias("accuracy_std"),
        pl.col("used_input_tokens").unique().item(),
        pl.col("used_output_tokens").unique().item(),
    )
)

In [75]:
all_group_combinations_majority_voting.head()

family,group,no_participants,sizes,model_names,accuracy_mean,accuracy_std,used_input_tokens,used_output_tokens
str,str,i64,list[str],list[str],f64,f64,f64,f64
"""Qwen3.5""","""LLM""",9,"[""L"", ""L"", ""M""]","[""Qwen/Qwen3.5-0.8B"", ""Qwen/Qwen3.5-0.8B"", ""Qwen/Qwen3.5-4B""]",0.493,0.008654,12551.997,18704.72925
"""Ministral""","""HHH""",6,"[""H"", ""H"", ""H""]","[""mistralai/Ministral-3-14B-Reasoning-2512"", ""mistralai/Ministral-3-14B-Reasoning-2512"", ""mistralai/Ministral-3-14B-Reasoning-2512""]",0.6839,0.003178,8096.952,3977.1975
"""Qwen3""","""LM""",12,"[""L"", ""M""]","[""Qwen/Qwen3-0.6B"", ""Qwen/Qwen3-4B""]",0.6099,0.004932,16345.8,17937.97
"""Ministral""","""HH""",2,"[""H"", ""H""]","[""mistralai/Ministral-3-14B-Reasoning-2512"", ""mistralai/Ministral-3-14B-Reasoning-2512""]",0.6409,0.00528,2698.984,1325.7325
"""Ministral""","""HM""",12,"[""H"", ""M""]","[""mistralai/Ministral-3-8B-Reasoning-2512"", ""mistralai/Ministral-3-14B-Reasoning-2512""]",0.6579,0.003213,16193.904,7764.1895


In [76]:
register_materialization(
    "all_group_combinations_majority_voting",
    all_group_combinations_majority_voting,
    "all_group_combinations_majority_voting",
    force=True,
)

Wrote to:  /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/report/final_no_discussion_voting/all_group_combinations_majority_voting.parquet


/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagstermill/manager.py:286: BetaWarning: Class `DagstermillExecutionContext` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  self.context = DagstermillExecutionContext(
